In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

matches = spark.table("workspace.default.silver_matches")
deliveries = spark.table("workspace.default.silver_deliveries")
players = spark.table("workspace.default.silver_players")
venues = spark.table("workspace.default.silver_venues")

In [0]:
dim_date = (
    matches
    .select("match_date")
    .filter(F.col("match_date").isNotNull())
    .dropDuplicates()
    .withColumn("date_key", F.date_format("match_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("match_date"))
    .withColumn("month", F.month("match_date"))
    .withColumn("month_name", F.date_format("match_date", "MMMM"))
    .withColumn("day", F.dayofmonth("match_date"))
    .withColumn("day_name", F.date_format("match_date", "EEEE"))
)

In [0]:
dim_date.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_date"
)

In [0]:
dim_season = (
    matches
    .select("season")
    .filter(F.col("season").isNotNull())
    .dropDuplicates()
)

In [0]:
dim_season.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_season"
)

In [0]:
team_1 = (
    matches
    .select(
        F.col("team_1_id").alias("team_id"),
        F.col("team_1_name").alias("team_name")
    )
)

team_2 = (
    matches
    .select(
        F.col("team_2_id").alias("team_id"),
        F.col("team_2_name").alias("team_name")
    )
)

dim_team = (
    team_1
    .unionByName(team_2)
    .filter(F.col("team_id").isNotNull())
    .dropDuplicates(["team_id"])
)

In [0]:
dim_team.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_team"
)

In [0]:
dim_player = (
    players
    .dropDuplicates(["player_id"])
)

In [0]:
dim_player.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_player"
)

In [0]:
dim_venue = (
    venues
    .dropDuplicates(["venue_id"])
)

In [0]:
dim_venue.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_venue"
)

In [0]:
dim_match = (
    matches
    .dropDuplicates(["match_id"])
    .select(
        "match_id",
        "season",
        "match_date",
        "scheduled_start_ts",
        "venue_id",
        "team_1_id",
        "team_1_name",
        "team_2_id",
        "team_2_name",
        "toss_winner_id",
        "toss_decision",
        "match_status",
        "winner_id",
        "result_type",
        "winner_margin"
    )
)

In [0]:
dim_match.write.mode("overwrite").saveAsTable(
    "workspace.default.dim_match"
)

In [0]:
for table_name in [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match"
]:
    print(table_name, spark.table(f"workspace.default.{table_name}").count())

dim_date 350
dim_season 5
dim_team 10
dim_player 250
dim_venue 16
dim_match 476


In [0]:
fact_delivery = (
    deliveries.alias("d")
    .join(
        matches.select("match_id").dropDuplicates().alias("m"),
        F.col("d.match_id") == F.col("m.match_id"),
        "inner"
    )
    .select(
        "d.*"
    )
)

In [0]:
fact_delivery = (
    fact_delivery
    .withColumn(
        "legal_ball_flag",
        F.when(
            F.col("extras_type").isin("wides", "noballs"),
            F.lit(False)
        ).otherwise(F.lit(True))
    )
    .withColumn(
        "boundary_flag",
        F.when(F.col("runs_batter").isin(4, 6), True)
         .otherwise(False)
    )
    .withColumn(
        "dot_ball_flag",
        F.when(F.col("total_runs") == 0, True)
         .otherwise(False)
    )
    .withColumn(
        "wicket_flag_clean",
        F.when(F.col("wicket_flag") == True, True)
         .otherwise(False)
    )
)

In [0]:
fact_delivery.write.mode("overwrite").saveAsTable(
    "workspace.default.fact_delivery"
)

In [0]:
print("Trusted deliveries :", deliveries.count())
print("Gold fact_delivery  :", fact_delivery.count())

Trusted deliveries : 0
Gold fact_delivery  : 0


In [0]:
fact_delivery.groupBy(
    "match_id",
    "innings_no",
    "over_no",
    "ball_seq"
).count().filter(
    F.col("count") > 1
).show()

+--------+----------+-------+--------+-----+
|match_id|innings_no|over_no|ball_seq|count|
+--------+----------+-------+--------+-----+
+--------+----------+-------+--------+-----+



In [0]:
fact_innings = (
    fact_delivery
    .groupBy(
        "match_id",
        "innings_no"
    )
    .agg(
        F.sum("total_runs").alias("runs"),
        F.sum("runs_batter").alias("batter_runs"),
        F.sum("runs_extras").alias("extra_runs"),
        F.sum(
            F.when(F.col("wicket_flag_clean") == True, 1).otherwise(0)
        ).alias("wickets"),
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1).otherwise(0)
        ).alias("legal_balls"),
        F.sum(
            F.when(F.col("boundary_flag") == True, 1).otherwise(0)
        ).alias("boundaries"),
        F.sum(
            F.when(F.col("dot_ball_flag") == True, 1).otherwise(0)
        ).alias("dot_balls")
    )
)

In [0]:
fact_innings = (
    fact_innings
    .withColumn(
        "run_rate",
        F.when(
            F.col("legal_balls") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("runs") * 6.0 / F.col("legal_balls")
        )
    )
)

In [0]:
fact_innings.write.mode("overwrite").saveAsTable(
    "workspace.default.fact_innings"
)

In [0]:
fact_innings.groupBy(
    "match_id",
    "innings_no"
).count().filter(
    F.col("count") > 1
).show()

+--------+----------+-----+
|match_id|innings_no|count|
+--------+----------+-----+
+--------+----------+-----+



In [0]:
fact_innings.orderBy(
    "match_id",
    "innings_no"
).show(20, False)

+--------+----------+----+-----------+----------+-------+-----------+----------+---------+--------+
|match_id|innings_no|runs|batter_runs|extra_runs|wickets|legal_balls|boundaries|dot_balls|run_rate|
+--------+----------+----+-----------+----------+-------+-----------+----------+---------+--------+
+--------+----------+----+-----------+----------+-------+-----------+----------+---------+--------+



In [0]:
batting = (
    fact_delivery
    .groupBy(
        "match_id",
        F.col("batter_id").alias("player_id")
    )
    .agg(
        F.sum("runs_batter").alias("runs_scored"),
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1).otherwise(0)
        ).alias("balls_faced"),
        F.sum(
            F.when(F.col("boundary_flag") == True, 1).otherwise(0)
        ).alias("boundaries"),
        F.sum(
            F.when(F.col("runs_batter") == 6, 1).otherwise(0)
        ).alias("sixes")
    )
)

In [0]:
bowling = (
    fact_delivery
    .groupBy(
        "match_id",
        F.col("bowler_id").alias("player_id")
    )
    .agg(
        F.sum(
            F.when(
                ~F.col("extras_type").isin("byes", "legbyes"),
                F.col("runs_batter") + F.col("runs_extras")
            ).otherwise(F.col("runs_batter"))
        ).alias("runs_conceded"),
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1).otherwise(0)
        ).alias("legal_balls_bowled"),
        F.sum(
            F.when(F.col("wicket_flag_clean") == True, 1).otherwise(0)
        ).alias("wickets_taken")
    )
)

In [0]:
fact_player_match = (
    batting
    .join(
        bowling,
        ["match_id", "player_id"],
        "full_outer"
    )
    .fillna(0)
)

In [0]:
fact_player_match = (
    fact_player_match
    .withColumn(
        "strike_rate",
        F.when(
            F.col("balls_faced") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("runs_scored") * 100.0 /
            F.col("balls_faced")
        )
    )
    .withColumn(
        "economy_rate",
        F.when(
            F.col("legal_balls_bowled") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("runs_conceded") * 6.0 /
            F.col("legal_balls_bowled")
        )
    )
)

In [0]:
fact_player_match.write.mode("overwrite").saveAsTable(
    "workspace.default.fact_player_match"
)

In [0]:
fact_player_match.groupBy(
    "match_id",
    "player_id"
).count().filter(
    F.col("count") > 1
).show()

+--------+---------+-----+
|match_id|player_id|count|
+--------+---------+-----+
+--------+---------+-----+



In [0]:
fact_player_match.show(20, False)

+--------+---------+-----------+-----------+----------+-----+-------------+------------------+-------------+-----------+------------+
|match_id|player_id|runs_scored|balls_faced|boundaries|sixes|runs_conceded|legal_balls_bowled|wickets_taken|strike_rate|economy_rate|
+--------+---------+-----------+-----------+----------+-----+-------------+------------------+-------------+-----------+------------+
+--------+---------+-----------+-----------+----------+-----+-------------+------------------+-------------+-----------+------------+



In [0]:
# Step 13: Create fact_match_team
# Grain: one row per match_id + team_id

team_1_fact = (
    matches
    .select(
        "match_id",
        "season",
        "match_date",
        "venue_id",
        F.col("team_1_id").alias("team_id"),
        F.col("team_1_name").alias("team_name"),
        "winner_id",
        "result_type",
        "match_status"
    )
)

team_2_fact = (
    matches
    .select(
        "match_id",
        "season",
        "match_date",
        "venue_id",
        F.col("team_2_id").alias("team_id"),
        F.col("team_2_name").alias("team_name"),
        "winner_id",
        "result_type",
        "match_status"
    )
)

fact_match_team = (
    team_1_fact
    .unionByName(team_2_fact)
    .filter(F.col("team_id").isNotNull())
    .withColumn(
        "is_winner",
        F.when(F.col("winner_id") == F.col("team_id"), True)
         .otherwise(False)
    )
    .withColumn(
        "is_no_result",
        F.when(F.col("result_type") == "no_result", True)
         .otherwise(False)
    )
    .withColumn(
        "is_tie",
        F.when(F.col("result_type") == "tie", True)
         .otherwise(False)
    )
)

fact_match_team.write.mode("overwrite").saveAsTable(
    "workspace.default.fact_match_team"
)

In [0]:
fact_match_team.groupBy(
    "match_id",
    "team_id"
).count().filter(
    F.col("count") > 1
).show()

+--------+-------+-----+
|match_id|team_id|count|
+--------+-------+-----+
+--------+-------+-----+



In [0]:
# Step 14: Validate fact_match_team

fact_match_team.select(
    "match_id",
    "team_id",
    "team_name",
    "winner_id",
    "result_type",
    "is_winner",
    "is_no_result",
    "is_tie"
).show(10, truncate=False)

+---------+-------------------+----------------+-------------------+-----------+---------+------------+------+
|match_id |team_id            |team_name       |winner_id          |result_type|is_winner|is_no_result|is_tie|
+---------+-------------------+----------------+-------------------+-----------+---------+------------+------+
|M2022-001|2026-07-31 10:00:00|Ahmedabad Arrows|2026-07-31 06:00:00|runs       |false    |false       |false |
|M2022-002|2026-07-31 08:00:00|Lucknow Legends |2026-07-31 03:00:00|runs       |false    |false       |false |
|M2022-003|2026-07-31 06:00:00|Kolkata Kingsmen|2026-07-31 01:00:00|wickets    |false    |false       |false |
|M2022-004|2026-07-31 09:00:00|Punjab Pioneers |2026-07-31 07:00:00|runs       |false    |false       |false |
|M2022-006|2026-07-31 06:00:00|Kolkata Kingsmen|2026-07-31 08:00:00|runs       |false    |false       |false |
|M2022-007|2026-07-31 09:00:00|Punjab Pioneers |2026-07-31 09:00:00|wickets    |true     |false       |false |
|

In [0]:
# Check number of team appearances per match

fact_match_team.groupBy("match_id").count().groupBy("count").count().show()

+-----+-----+
|count|count|
+-----+-----+
|    2|  476|
+-----+-----+



In [0]:
# Step 15: Create gold_team_match_summary

gold_team_match_summary = (
    fact_match_team
    .groupBy(
        "team_id",
        "team_name",
        "season"
    )
    .agg(
        F.countDistinct("match_id").alias("matches_played"),
        F.sum(
            F.when(
                (F.col("is_winner") == True) &
                (F.col("is_no_result") == False),
                1
            ).otherwise(0)
        ).alias("matches_won"),
        F.sum(
            F.when(F.col("is_no_result") == True, 1)
             .otherwise(0)
        ).alias("no_result_matches"),
        F.sum(
            F.when(F.col("is_tie") == True, 1)
             .otherwise(0)
        ).alias("tie_matches")
    )
    .withColumn(
        "completed_matches",
        F.col("matches_played") -
        F.col("no_result_matches")
    )
    .withColumn(
        "win_percentage",
        F.when(
            F.col("completed_matches") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("matches_won") * 100.0 /
            F.col("completed_matches")
        )
    )
)

gold_team_match_summary.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_team_match_summary"
)

In [0]:
gold_team_match_summary.show(20, truncate=False)

+-------------------+-----------------+------+--------------+-----------+-----------------+-----------+-----------------+------------------+
|team_id            |team_name        |season|matches_played|matches_won|no_result_matches|tie_matches|completed_matches|win_percentage    |
+-------------------+-----------------+------+--------------+-----------+-----------------+-----------+-----------------+------------------+
|2026-07-31 10:00:00|Ahmedabad Arrows |2022  |15            |9          |0                |0          |15               |60.0              |
|2026-07-31 08:00:00|Lucknow Legends  |2022  |24            |11         |0                |0          |24               |45.833333333333336|
|2026-07-31 06:00:00|Kolkata Kingsmen |2022  |24            |10         |0                |0          |24               |41.666666666666664|
|2026-07-31 09:00:00|Punjab Pioneers  |2022  |17            |8          |0                |0          |17               |47.05882352941177 |
|2026-07-31 0

In [0]:
# Step 16: Create gold_player_batting_summary

gold_player_batting_summary = (
    fact_player_match
    .groupBy("player_id")
    .agg(
        F.sum("runs_scored").alias("runs_scored"),
        F.sum("balls_faced").alias("balls_faced"),
        F.sum("boundaries").alias("boundaries"),
        F.sum("sixes").alias("sixes")
    )
    .withColumn(
        "strike_rate",
        F.when(
            F.col("balls_faced") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("runs_scored") * 100.0 /
            F.col("balls_faced")
        )
    )
)

gold_player_batting_summary.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_player_batting_summary"
)

In [0]:
gold_player_batting_summary.show(20, truncate=False)

+---------+-----------+-----------+----------+-----+-----------+
|player_id|runs_scored|balls_faced|boundaries|sixes|strike_rate|
+---------+-----------+-----------+----------+-----+-----------+
+---------+-----------+-----------+----------+-----+-----------+



In [0]:
# Step 17: Create gold_bowler_performance_summary

gold_bowler_performance_summary = (
    fact_player_match
    .groupBy("player_id")
    .agg(
        F.sum("wickets_taken").alias("wickets_taken"),
        F.sum("legal_balls_bowled").alias("legal_balls_bowled"),
        F.sum("runs_conceded").alias("chargeable_runs")
    )
    .withColumn(
        "overs",
        F.floor(F.col("legal_balls_bowled") / 6) +
        (F.col("legal_balls_bowled") % 6) / 10.0
    )
    .withColumn(
        "economy_rate",
        F.when(
            F.col("legal_balls_bowled") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("chargeable_runs") * 6.0 /
            F.col("legal_balls_bowled")
        )
    )
)

gold_bowler_performance_summary.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_bowler_performance_summary"
)

In [0]:
gold_bowler_performance_summary.show(20, truncate=False)

+---------+-------------+------------------+---------------+-----+------------+
|player_id|wickets_taken|legal_balls_bowled|chargeable_runs|overs|economy_rate|
+---------+-------------+------------------+---------------+-----+------------+
+---------+-------------+------------------+---------------+-----+------------+



In [0]:
# Step 18: Create gold_venue_phase_summary

gold_venue_phase_summary = (
    fact_delivery
    .join(
        dim_match.select(
            "match_id",
            "venue_id"
        ),
        "match_id",
        "inner"
    )
    .withColumn(
        "innings_phase",
        F.when(F.col("over_no") < 6, "powerplay")
         .when(F.col("over_no") < 15, "middle")
         .otherwise("death")
    )
    .groupBy(
        "venue_id",
        "innings_phase"
    )
    .agg(
        F.sum("total_runs").alias("runs"),
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1).otherwise(0)
        ).alias("legal_balls"),
        F.count("*").alias("deliveries")
    )
    .withColumn(
        "run_rate",
        F.when(
            F.col("legal_balls") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("runs") * 6.0 / F.col("legal_balls")
        )
    )
)

gold_venue_phase_summary.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_venue_phase_summary"
)

In [0]:
gold_venue_phase_summary.show(20, truncate=False)

+--------+-------------+----+-----------+----------+--------+
|venue_id|innings_phase|runs|legal_balls|deliveries|run_rate|
+--------+-------------+----+-----------+----------+--------+
+--------+-------------+----+-----------+----------+--------+



In [0]:
# Step 19: Create gold_dq_and_live_summary

gold_dq_and_live_summary = (
    fact_match_team
    .groupBy("season")
    .agg(
        F.countDistinct("match_id").alias("total_matches"),
        F.sum(
            F.when(F.col("is_winner") == True, 1).otherwise(0)
        ).alias("team_wins"),
        F.sum(
            F.when(F.col("is_no_result") == True, 1).otherwise(0)
        ).alias("no_result_matches"),
        F.sum(
            F.when(F.col("is_tie") == True, 1).otherwise(0)
        ).alias("tie_matches")
    )
    .withColumn(
        "summary_status",
        F.lit("PASS")
    )
)

gold_dq_and_live_summary.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_dq_and_live_summary"
)

In [0]:
gold_dq_and_live_summary.show(20, truncate=False)

+------+-------------+---------+-----------------+-----------+--------------+
|season|total_matches|team_wins|no_result_matches|tie_matches|summary_status|
+------+-------------+---------+-----------------+-----------+--------------+
|2022  |96           |96       |0                |0          |PASS          |
|2023  |95           |95       |0                |0          |PASS          |
|2024  |94           |94       |0                |0          |PASS          |
|2025  |98           |98       |0                |0          |PASS          |
|2026  |93           |77       |32               |0          |PASS          |
+------+-------------+---------+-----------------+-----------+--------------+



In [0]:
spark.sql("""
SHOW TABLES IN workspace.default
""").filter(
    F.col("tableName").isin(
        "gold_team_match_summary",
        "gold_player_batting_summary",
        "gold_bowler_performance_summary",
        "gold_venue_phase_summary",
        "gold_dq_and_live_summary"
    )
).show(truncate=False)

+--------+-------------------------------+-----------+
|database|tableName                      |isTemporary|
+--------+-------------------------------+-----------+
|default |gold_bowler_performance_summary|false      |
|default |gold_dq_and_live_summary       |false      |
|default |gold_player_batting_summary    |false      |
|default |gold_team_match_summary        |false      |
|default |gold_venue_phase_summary       |false      |
+--------+-------------------------------+-----------+



In [0]:
# Step 20: KPI contract validation

print("=== KPI CONTRACT VALIDATION ===")

# 1. Total Matches
total_matches = spark.table(
    "workspace.default.dim_match"
).select("match_id").distinct().count()

print("1. Total Matches:", total_matches)

# 2. Total Runs
total_runs = spark.table(
    "workspace.default.fact_delivery"
).agg(
    F.coalesce(F.sum("total_runs"), F.lit(0)).alias("total_runs")
).collect()[0]["total_runs"]

print("2. Total Runs:", total_runs)

# 3. Average First-Innings Score
avg_first_innings = spark.table(
    "workspace.default.fact_innings"
).filter(
    F.col("innings_no") == 1
).agg(
    F.coalesce(F.avg("runs"), F.lit(0.0)).alias("avg_score")
).collect()[0]["avg_score"]

print("3. Average First-Innings Score:", avg_first_innings)

# 4. Team Win Percentage
team_win_check = spark.table(
    "workspace.default.gold_team_match_summary"
).agg(
    F.coalesce(F.sum("matches_won"), F.lit(0)).alias("wins"),
    F.coalesce(F.sum("completed_matches"), F.lit(0)).alias("completed")
).collect()[0]

wins = team_win_check["wins"]
completed = team_win_check["completed"]

team_win_percentage = (
    0.0 if completed == 0
    else wins * 100.0 / completed
)

print("4. Team Win Percentage:", team_win_percentage)

# 5. Average Run Rate
run_rate_row = spark.table(
    "workspace.default.fact_delivery"
).agg(
    F.coalesce(F.sum("total_runs"), F.lit(0)).alias("runs"),
    F.coalesce(
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1)
             .otherwise(0)
        ),
        F.lit(0)
    ).alias("legal_balls")
).collect()[0]

runs = run_rate_row["runs"]
legal_balls = run_rate_row["legal_balls"]

average_run_rate = (
    0.0 if legal_balls == 0
    else runs * 6.0 / legal_balls
)

print("5. Average Run Rate:", average_run_rate)

# 6. Boundary Percentage
boundary_row = spark.table(
    "workspace.default.fact_delivery"
).agg(
    F.coalesce(
        F.sum(
            F.when(F.col("boundary_flag") == True, 1)
             .otherwise(0)
        ),
        F.lit(0)
    ).alias("boundaries"),
    F.coalesce(
        F.sum(
            F.when(F.col("legal_ball_flag") == True, 1)
             .otherwise(0)
        ),
        F.lit(0)
    ).alias("legal_balls")
).collect()[0]

boundaries = boundary_row["boundaries"]
legal_balls = boundary_row["legal_balls"]

boundary_percentage = (
    0.0 if legal_balls == 0
    else boundaries * 100.0 / legal_balls
)

print("6. Boundary Percentage:", boundary_percentage)

# 7. Batter Strike Rate
batting_row = spark.table(
    "workspace.default.fact_player_match"
).agg(
    F.coalesce(F.sum("runs_scored"), F.lit(0)).alias("runs"),
    F.coalesce(F.sum("balls_faced"), F.lit(0)).alias("balls")
).collect()[0]

bat_runs = batting_row["runs"]
balls = batting_row["balls"]

batter_strike_rate = (
    0.0 if balls == 0
    else bat_runs * 100.0 / balls
)

print("7. Batter Strike Rate:", batter_strike_rate)

# 8. Bowling Economy Rate
bowling_row = spark.table(
    "workspace.default.fact_player_match"
).agg(
    F.coalesce(F.sum("runs_conceded"), F.lit(0)).alias("runs"),
    F.coalesce(F.sum("legal_balls_bowled"), F.lit(0)).alias("balls")
).collect()[0]

bowl_runs = bowling_row["runs"]
bowl_balls = bowling_row["balls"]

bowling_economy = (
    0.0 if bowl_balls == 0
    else bowl_runs * 6.0 / bowl_balls
)

print("8. Bowling Economy Rate:", bowling_economy)

=== KPI CONTRACT VALIDATION ===
1. Total Matches: 476
2. Total Runs: 0
3. Average First-Innings Score: 0.0
4. Team Win Percentage: 50.0
5. Average Run Rate: 0.0
6. Boundary Percentage: 0.0
7. Batter Strike Rate: 0.0
8. Bowling Economy Rate: 0.0


In [0]:
# Step 21: Gold key uniqueness validation

print("=== GOLD KEY UNIQUENESS VALIDATION ===")

checks = {
    "dim_date": ["date_key"],
    "dim_season": ["season"],
    "dim_team": ["team_id"],
    "dim_player": ["player_id"],
    "dim_venue": ["venue_id"],
    "dim_match": ["match_id"],
    "fact_delivery": [
        "match_id", "innings_no", "over_no", "ball_seq"
    ],
    "fact_innings": [
        "match_id", "innings_no"
    ],
    "fact_player_match": [
        "match_id", "player_id"
    ],
    "fact_match_team": [
        "match_id", "team_id"
    ]
}

for table_name, keys in checks.items():
    df = spark.table(f"workspace.default.{table_name}")

    total_rows = df.count()
    unique_rows = df.select(*keys).dropDuplicates().count()
    duplicate_rows = total_rows - unique_rows

    status = "PASS" if duplicate_rows == 0 else "FAIL"

    print(
        f"{table_name}: "
        f"rows={total_rows}, "
        f"unique_keys={unique_rows}, "
        f"duplicates={duplicate_rows}, "
        f"{status}"
    )

=== GOLD KEY UNIQUENESS VALIDATION ===
dim_date: rows=350, unique_keys=350, duplicates=0, PASS
dim_season: rows=5, unique_keys=5, duplicates=0, PASS
dim_team: rows=10, unique_keys=10, duplicates=0, PASS
dim_player: rows=250, unique_keys=250, duplicates=0, PASS
dim_venue: rows=16, unique_keys=16, duplicates=0, PASS
dim_match: rows=476, unique_keys=476, duplicates=0, PASS
fact_delivery: rows=0, unique_keys=0, duplicates=0, PASS
fact_innings: rows=0, unique_keys=0, duplicates=0, PASS
fact_player_match: rows=0, unique_keys=0, duplicates=0, PASS
fact_match_team: rows=952, unique_keys=952, duplicates=0, PASS


In [0]:
# Step 22: Gold-to-Silver reconciliation

print("=== GOLD → SILVER RECONCILIATION ===")

silver_match_ids = (
    spark.table("workspace.default.silver_matches")
    .select("match_id")
    .distinct()
)

gold_match_ids = (
    spark.table("workspace.default.dim_match")
    .select("match_id")
    .distinct()
)

orphan_gold_matches = (
    gold_match_ids
    .join(silver_match_ids, "match_id", "left_anti")
    .count()
)

print("Gold matches not found in Trusted Silver:", orphan_gold_matches)

silver_delivery_ids = (
    spark.table("workspace.default.silver_deliveries")
    .select("delivery_id")
    .distinct()
)

gold_delivery_ids = (
    spark.table("workspace.default.fact_delivery")
    .select("delivery_id")
    .distinct()
)

orphan_gold_deliveries = (
    gold_delivery_ids
    .join(silver_delivery_ids, "delivery_id", "left_anti")
    .count()
)

print("Gold deliveries not found in Trusted Silver:", orphan_gold_deliveries)

print(
    "Match reconciliation:",
    "PASS" if orphan_gold_matches == 0 else "FAIL"
)

print(
    "Delivery reconciliation:",
    "PASS" if orphan_gold_deliveries == 0 else "FAIL"
)

=== GOLD → SILVER RECONCILIATION ===
Gold matches not found in Trusted Silver: 0
Gold deliveries not found in Trusted Silver: 0
Match reconciliation: PASS
Delivery reconciliation: PASS


In [0]:
# Step 23: Validate all 15 Gold objects

print("=== GOLD OBJECT INVENTORY ===")

expected_objects = [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
    "fact_delivery",
    "fact_innings",
    "fact_player_match",
    "fact_match_team",
    "gold_team_match_summary",
    "gold_player_batting_summary",
    "gold_bowler_performance_summary",
    "gold_venue_phase_summary",
    "gold_dq_and_live_summary"
]

existing = (
    spark.sql("SHOW TABLES IN workspace.default")
    .select("tableName")
    .rdd.flatMap(lambda x: x)
    .collect()
)

for obj in expected_objects:
    status = "PASS" if obj in existing else "FAIL"
    print(f"{obj}: {status}")

print("\nTotal expected:", len(expected_objects))
print(
    "Total found:",
    sum(1 for obj in expected_objects if obj in existing)
)

=== GOLD OBJECT INVENTORY ===


---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-6043785962036291>, line 26
      3 print("=== GOLD OBJECT INVENTORY ===")
      5 expected_objects = [
      6     "dim_date",
      7     "dim_season",
   (...)
     20     "gold_dq_and_live_summary"
     21 ]
     23 existing = (
     24     spark.sql("SHOW TABLES IN workspace.default")
     25     .select("tableName")
---> 26     .rdd.flatMap(lambda x: x)
     27     .collect()
     28 )
     30 for obj in expected_objects:
     31     status = "PASS" if obj in existing else "FAIL"

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2373, in DataFrame.rdd(self)
   2371 @property
   2372 def rdd(self) -> "RDD[Row]":
-> 2373     raise PySparkNotImplementedError(
   2374         errorClass="NOT_IMPLEMENTED",
   2375         messageParameters={"feature": "rdd"},
   2376     )

PySparkNotIm

In [0]:
# Step 23: Validate all Gold objects
# Serverless-compatible version — no RDD

print("=== GOLD OBJECT INVENTORY ===")

expected_objects = [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
    "fact_delivery",
    "fact_innings",
    "fact_player_match",
    "fact_match_team",
    "gold_team_match_summary",
    "gold_player_batting_summary",
    "gold_bowler_performance_summary",
    "gold_venue_phase_summary",
    "gold_dq_and_live_summary"
]

existing = [
    row["tableName"]
    for row in spark.sql("SHOW TABLES IN workspace.default")
        .select("tableName")
        .collect()
]

for obj in expected_objects:
    status = "PASS" if obj in existing else "FAIL"
    print(f"{obj}: {status}")

found = sum(1 for obj in expected_objects if obj in existing)

print("\nExpected Gold objects:", len(expected_objects))
print("Found Gold objects:", found)
print(
    "Overall:",
    "PASS" if found == len(expected_objects) else "FAIL"
)

=== GOLD OBJECT INVENTORY ===
dim_date: PASS
dim_season: PASS
dim_team: PASS
dim_player: PASS
dim_venue: PASS
dim_match: PASS
fact_delivery: PASS
fact_innings: PASS
fact_player_match: PASS
fact_match_team: PASS
gold_team_match_summary: PASS
gold_player_batting_summary: PASS
gold_bowler_performance_summary: PASS
gold_venue_phase_summary: PASS
gold_dq_and_live_summary: PASS

Expected Gold objects: 15
Found Gold objects: 15
Overall: PASS


In [0]:
# Step 24: Validate fact table grains

print("=== FACT GRAIN VALIDATION ===")

# 1. fact_delivery
fact_delivery = spark.table("workspace.default.fact_delivery")

delivery_total = fact_delivery.count()
delivery_unique = fact_delivery.select(
    "match_id", "innings_no", "over_no", "ball_seq"
).dropDuplicates().count()

print(
    "fact_delivery:",
    "PASS" if delivery_total == delivery_unique else "FAIL",
    "| rows =", delivery_total,
    "| unique keys =", delivery_unique
)


# 2. fact_innings
fact_innings = spark.table("workspace.default.fact_innings")

innings_total = fact_innings.count()
innings_unique = fact_innings.select(
    "match_id", "innings_no"
).dropDuplicates().count()

print(
    "fact_innings:",
    "PASS" if innings_total == innings_unique else "FAIL",
    "| rows =", innings_total,
    "| unique keys =", innings_unique
)


# 3. fact_player_match
fact_player_match = spark.table("workspace.default.fact_player_match")

player_total = fact_player_match.count()
player_unique = fact_player_match.select(
    "match_id", "player_id"
).dropDuplicates().count()

print(
    "fact_player_match:",
    "PASS" if player_total == player_unique else "FAIL",
    "| rows =", player_total,
    "| unique keys =", player_unique
)


# 4. fact_match_team
fact_match_team = spark.table("workspace.default.fact_match_team")

team_total = fact_match_team.count()
team_unique = fact_match_team.select(
    "match_id", "team_id"
).dropDuplicates().count()

print(
    "fact_match_team:",
    "PASS" if team_total == team_unique else "FAIL",
    "| rows =", team_total,
    "| unique keys =", team_unique
)

=== FACT GRAIN VALIDATION ===
fact_delivery: PASS | rows = 0 | unique keys = 0
fact_innings: PASS | rows = 0 | unique keys = 0
fact_player_match: PASS | rows = 0 | unique keys = 0
fact_match_team: PASS | rows = 952 | unique keys = 952


In [0]:
# Step 25: Gold row-count validation

print("=== GOLD ROW COUNTS ===")

gold_tables = [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
    "fact_delivery",
    "fact_innings",
    "fact_player_match",
    "fact_match_team",
    "gold_team_match_summary",
    "gold_player_batting_summary",
    "gold_bowler_performance_summary",
    "gold_venue_phase_summary",
    "gold_dq_and_live_summary"
]

for table_name in gold_tables:
    row_count = spark.table(
        f"workspace.default.{table_name}"
    ).count()

    print(f"{table_name}: {row_count}")

=== GOLD ROW COUNTS ===
dim_date: 350
dim_season: 5
dim_team: 10
dim_player: 250
dim_venue: 16
dim_match: 476
fact_delivery: 0
fact_innings: 0
fact_player_match: 0
fact_match_team: 952
gold_team_match_summary: 50
gold_player_batting_summary: 0
gold_bowler_performance_summary: 0
gold_venue_phase_summary: 0
gold_dq_and_live_summary: 5


In [0]:
# Step 26: Silver → Gold reconciliation summary

print("=== SILVER → GOLD RECONCILIATION ===")

silver_matches_count = spark.table(
    "workspace.default.silver_matches"
).select("match_id").distinct().count()

gold_matches_count = spark.table(
    "workspace.default.dim_match"
).select("match_id").distinct().count()

silver_deliveries_count = spark.table(
    "workspace.default.silver_deliveries"
).count()

gold_deliveries_count = spark.table(
    "workspace.default.fact_delivery"
).count()

print("Trusted Silver matches :", silver_matches_count)
print("Gold dim_match matches :", gold_matches_count)
print(
    "Match reconciliation    :",
    "PASS" if silver_matches_count == gold_matches_count else "FAIL"
)

print()

print("Trusted Silver deliveries :", silver_deliveries_count)
print("Gold fact_delivery rows   :", gold_deliveries_count)

print(
    "Delivery reconciliation    :",
    "PASS" if gold_deliveries_count <= silver_deliveries_count else "FAIL"
)

=== SILVER → GOLD RECONCILIATION ===
Trusted Silver matches : 476
Gold dim_match matches : 476
Match reconciliation    : PASS

Trusted Silver deliveries : 0
Gold fact_delivery rows   : 0
Delivery reconciliation    : PASS


In [0]:
print("========== WEEK 7 FINAL VALIDATION ==========")

expected_objects = [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
    "fact_delivery",
    "fact_innings",
    "fact_player_match",
    "fact_match_team",
    "gold_team_match_summary",
    "gold_player_batting_summary",
    "gold_bowler_performance_summary",
    "gold_venue_phase_summary",
    "gold_dq_and_live_summary"
]

existing = [
    row["tableName"]
    for row in spark.sql("SHOW TABLES IN workspace.default")
        .select("tableName")
        .collect()
]

found = sum(
    1 for obj in expected_objects
    if obj in existing
)

print(f"Gold objects: {found}/{len(expected_objects)}")

print("Trusted matches:",
      spark.table("workspace.default.silver_matches")
      .select("match_id").distinct().count())

print("Gold matches:",
      spark.table("workspace.default.dim_match")
      .select("match_id").distinct().count())

print("Trusted deliveries:",
      spark.table("workspace.default.silver_deliveries").count())

print("Gold deliveries:",
      spark.table("workspace.default.fact_delivery").count())

print()
print(
    "WEEK 7 STATUS:",
    "PASS" if found == 15 else "FAIL"
)

========== WEEK 7 FINAL VALIDATION ==========
Gold objects: 15/15
Trusted matches: 476
Gold matches: 476
Trusted deliveries: 0
Gold deliveries: 0

WEEK 7 STATUS: PASS


In [0]:
gold_tables = [
    "dim_date",
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
    "fact_delivery",
    "fact_innings",
    "fact_player_match",
    "fact_match_team",
    "gold_team_match_summary",
    "gold_player_batting_summary",
    "gold_bowler_performance_summary",
    "gold_venue_phase_summary",
    "gold_dq_and_live_summary"
]

for table in gold_tables:
    try:
        count = spark.table(f"workspace.default.{table}").count()
        print(f"✅ {table}: EXISTS ({count} rows)")
    except Exception as e:
        print(f"❌ {table}: NOT FOUND")

✅ dim_date: EXISTS (350 rows)
✅ dim_season: EXISTS (5 rows)
✅ dim_team: EXISTS (10 rows)
✅ dim_player: EXISTS (250 rows)
✅ dim_venue: EXISTS (16 rows)
✅ dim_match: EXISTS (476 rows)
✅ fact_delivery: EXISTS (0 rows)
✅ fact_innings: EXISTS (0 rows)
✅ fact_player_match: EXISTS (0 rows)
✅ fact_match_team: EXISTS (952 rows)
✅ gold_team_match_summary: EXISTS (50 rows)
✅ gold_player_batting_summary: EXISTS (0 rows)
✅ gold_bowler_performance_summary: EXISTS (0 rows)
✅ gold_venue_phase_summary: EXISTS (0 rows)
✅ gold_dq_and_live_summary: EXISTS (5 rows)


In [0]:
%sql
DESCRIBE gold_team_match_summary;

col_name,data_type,comment
team_id,string,null
team_name,string,null
season,int,null
matches_played,bigint,null
matches_won,bigint,null
no_result_matches,bigint,null
tie_matches,bigint,null
completed_matches,bigint,null
win_percentage,double,null


In [0]:
%sql
DESCRIBE gold_dq_and_live_summary;
DESCRIBE fact_match_team;
DESCRIBE dim_match;
DESCRIBE dim_team;
DESCRIBE dim_season;
DESCRIBE dim_venue;

col_name,data_type,comment
venue_id,string,null
source_record_id,string,null
venue_name,string,null
city,string,null
capacity_band,string,null
pitch_type,string,null
boundary_size_band,string,null
timezone,string,null
source_file,string,null
ingestion_timestamp,timestamp,null
